# MODIS LST Day & Night — Porto Alegre (MOD11A2)

**Dataset:** MODIS Terra Land Surface Temperature/Emissivity 8-Day L3 Global 1 km (`MODIS/061/MOD11A2`)  
**Bands:** `LST_Day_1km` · `LST_Night_1km`  
**Resolution:** ~1 km  
**Period:** DJF seasons 2015–2024  
**Products:** P90 composite for Day and Night → two heat hazard layers

## Why two separate layers?

| Layer | Overpass (local) | What it captures | Hazard relevance |
|-------|-----------------|------------------|------------------|
| **LST Day** | ~10:30 | Peak surface radiative temperature | Intermediate-resolution thermal signal; bridges Landsat (30 m) and ERA5 (9 km) |
| **LST Night** | ~22:30 | Overnight temperature — city's ability to cool down | **Critical for health:** persistent nighttime heat prevents physiological recovery; key driver of heat-mortality |

## Processing steps (same for Day and Night)
1. Filter `MODIS/061/MOD11A2` to DJF months (Dec–Feb), 2015–2024, POA bounds.
2. Apply scale factor: `LST_K = DN × 0.02`; convert to °C (`LST_°C = LST_K − 273.15`).
3. Mask poor-quality pixels using `QC_Day` / `QC_Night` bits 0–1  
   (keep `00` = good quality and `01` = acceptable; exclude `10` = cloud, `11` = other).
4. Reduce to **P90 per pixel** across all valid DJF composites (2015–2024).
5. Count valid observations per pixel (QA layer).
6. Compute **LST anomaly** relative to the POA mean.
7. **Min-max normalize** to 0–1 (input for heat hazard ensemble).

## Outputs
| File | Description |
|------|-------------|
| `mod11a2_lst_day_p90_djf_2015_2024_poa.tif` | P90 daytime LST (°C), 1 km |
| `mod11a2_lst_night_p90_djf_2015_2024_poa.tif` | P90 nighttime LST (°C), 1 km |
| `mod11a2_lst_day_norm_djf_2015_2024_poa.tif` | Normalized daytime LST (0–1) |
| `mod11a2_lst_night_norm_djf_2015_2024_poa.tif` | Normalized nighttime LST (0–1) |

In [8]:
# Site configuration — uses transformation/heat_hazard city configs
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_HEAT_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "heat_hazard" if _candidate.name != "heat_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _HEAT_HAZARD = _probe
        break
if _HEAT_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/heat_hazard from notebook cwd")

sys.path.insert(0, str(_HEAT_HAZARD))
from site_config import load_site_config

HEAT_HAZARD_ROOT = _HEAT_HAZARD
HEAT_ROOT = HEAT_HAZARD_ROOT  # backward-compatible alias

# Set the city here (edit this line). That value wins for interactive runs.
# Use None to fall back to env HEAT_SITE (default porto_alegre).
SITE_SLUG = "plymouth"  # or: "porto_alegre" | "edina" | "richfield" | "rochester" | "apple_valley" | None
if SITE_SLUG is None:
    SITE_SLUG = os.environ.get("HEAT_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, HEAT_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
SEASON = SITE_CONFIG["season"]
SEASON_LABEL = SITE_CONFIG["season_label"]
START_YEAR = int(SITE_CONFIG["start_year"])
END_YEAR = int(SITE_CONFIG["end_year"])
print(f"Heat hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Season: {SEASON_LABEL} {START_YEAR}-{END_YEAR}")
print(f"Inputs -> {INPUT_DIR}")


Heat hazard site: Plymouth (plymouth)
Config: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/config/sites/plymouth.yaml
Season: JJA 2015-2024
Inputs -> /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input


## 0. Setup

In [9]:
import ee
import geemap
import numpy as np

ee.Authenticate()
ee.Initialize(
    project='eecc-maureen',
    opt_url='https://earthengine-highvolume.googleapis.com'
)

## 1. Area of interest 

In [10]:
# Site ROI: use the site polygon when available; fall back to bbox.
import json


def load_site_roi() -> ee.Geometry:
    boundary_path = SITE_CONFIG["boundary_path_abs"]
    if boundary_path.exists():
        data = json.loads(boundary_path.read_text())
        if data.get("type") == "FeatureCollection":
            features = [
                ee.Feature(ee.Geometry(feature["geometry"]), feature.get("properties", {}))
                for feature in data.get("features", [])
                if feature.get("geometry")
            ]
            if features:
                return ee.FeatureCollection(features).geometry()
        if data.get("type") == "Feature":
            return ee.Geometry(data["geometry"])
        if data.get("type") in {"Polygon", "MultiPolygon", "GeometryCollection"}:
            return ee.Geometry(data)
    return ee.Geometry.Rectangle(SITE_CONFIG["bbox"])


roi = load_site_roi()
print(f"ROI loaded for {SITE_CONFIG['display_name']} from {SITE_CONFIG['boundary_path_abs']}")


ROI loaded for Plymouth from /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/boundary/site.geojson


## 2. Load and filter MOD11A2

MOD11A2 is an 8-day composite product.  
The seasonal filter uses the configured site season (`SEASON`): `JJA` for Minnesota, `DJF` for Porto Alegre, or another supported season from `SEASON_MONTHS`.

In [11]:
START_DATE = f'{START_YEAR}-01-01'
END_DATE   = f'{END_YEAR}-12-31'

SEASON_MONTHS = {
    'djf': [12, 1, 2],
    'mam': [3, 4, 5],
    'jja': [6, 7, 8],
    'son': [9, 10, 11],
    'annual': list(range(1, 13)),
}
months = SEASON_MONTHS.get(SEASON.lower())
if months is None:
    raise ValueError(f'Unsupported season {SEASON!r}. Expected one of {sorted(SEASON_MONTHS)}')

month_filter = ee.Filter.inList('month', months)

raw = (
    ee.ImageCollection('MODIS/061/MOD11A2')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(roi)
    .map(lambda image: image.set('month', image.date().get('month')))
    .filter(month_filter)
    .select(['LST_Day_1km', 'QC_Day', 'LST_Night_1km', 'QC_Night'])
)

print(f'Season months ({SEASON_LABEL}):', months)
print('8-day composites in collection:', raw.size().getInfo())

Season months (JJA): [6, 7, 8]
8-day composites in collection: 120


## 3. Scale factors and quality mask

**LST conversion:**
```
LST_K   = DN × 0.02
LST_°C  = LST_K − 273.15
```

**QC_Day / QC_Night bits 0–1 (Mandatory QA):**
| Value | Meaning | Keep? |
|-------|---------|-------|
| `00` | Good quality, no further QA needed | ✓ |
| `01` | Other quality, acceptable | ✓ |
| `10` | LST not produced — cloud | ✗ |
| `11` | LST not produced — other reasons | ✗ |

In [5]:
def preprocess_modis(image):
    """
    Apply scale factor, convert to Celsius, and mask poor-quality pixels
    independently for Day and Night LST.
    """
    # ── Day ──────────────────────────────────────────────────────────────────
    lst_day = (
        image.select('LST_Day_1km')
        .multiply(0.02)
        .subtract(273.15)
        .rename('lst_day_celsius')
    )
    qc_day   = image.select('QC_Day')
    good_day = qc_day.bitwiseAnd(0b11).lt(2)   # keep 00 and 01
    lst_day  = lst_day.updateMask(good_day)

    # ── Night ─────────────────────────────────────────────────────────────────
    lst_night = (
        image.select('LST_Night_1km')
        .multiply(0.02)
        .subtract(273.15)
        .rename('lst_night_celsius')
    )
    qc_night   = image.select('QC_Night')
    good_night = qc_night.bitwiseAnd(0b11).lt(2)
    lst_night  = lst_night.updateMask(good_night)

    return image.addBands(lst_day).addBands(lst_night)


collection = raw.map(preprocess_modis)
print('Preprocessed collection size:', collection.size().getInfo())

Preprocessed collection size: 120


## 4. P90 composites — Day and Night

90th-percentile LST over all valid DJF composites (2015–2024) → chronic high-heat condition.

In [6]:
day_col   = collection.select('lst_day_celsius')
night_col = collection.select('lst_night_celsius')

lst_day_p90   = day_col.reduce(ee.Reducer.percentile([90])).rename('lst_day_p90').clip(roi)
lst_night_p90 = night_col.reduce(ee.Reducer.percentile([90])).rename('lst_night_p90').clip(roi)

day_obs   = day_col.reduce(ee.Reducer.count()).rename('day_obs_count').clip(roi)
night_obs = night_col.reduce(ee.Reducer.count()).rename('night_obs_count').clip(roi)


def get_stats(image, band, label):
    s = image.reduceRegion(
        reducer=ee.Reducer.min()
            .combine(ee.Reducer.mean(), sharedInputs=True)
            .combine(ee.Reducer.max(),  sharedInputs=True),
        geometry=roi, scale=1000, maxPixels=1e8, bestEffort=True,
    ).getInfo()
    print(f'  {label}: min={s[band+"_min"]:.2f}  mean={s[band+"_mean"]:.2f}  max={s[band+"_max"]:.2f} °C')


print('P90 LST composite stats:')
get_stats(lst_day_p90,   'lst_day_p90',   'LST Day   P90')
get_stats(lst_night_p90, 'lst_night_p90', 'LST Night P90')

P90 LST composite stats:
  LST Day   P90: min=15.56  mean=28.57  max=40.40 °C
  LST Night P90: min=0.27  mean=17.30  max=32.87 °C


## 5. Observation count QA

With 8-day composites and DJF filtering, each pixel should have ≥ ~8 valid observations.  
Day and night counts are checked separately because cloud-masking may differ.

In [7]:
def obs_stats(image, band, label):
    s = image.reduceRegion(
        reducer=ee.Reducer.percentile([5, 25, 50, 75, 95])
            .combine(ee.Reducer.min(), sharedInputs=True)
            .combine(ee.Reducer.max(), sharedInputs=True),
        geometry=roi, scale=1000, maxPixels=1e8, bestEffort=True,
    ).getInfo()
    print(f'  {label}')
    for k in sorted(s):
        print(f'    {k}: {s[k]:.1f}')


print('Observation counts per pixel:')
obs_stats(day_obs,   'day_obs_count',   'Day')
obs_stats(night_obs, 'night_obs_count', 'Night')

Observation counts per pixel:
  Day
    day_obs_count_max: 120.0
    day_obs_count_min: 102.0
    day_obs_count_p25: 119.0
    day_obs_count_p5: 117.0
    day_obs_count_p50: 119.0
    day_obs_count_p75: 120.0
    day_obs_count_p95: 120.0
  Night
    night_obs_count_max: 120.0
    night_obs_count_min: 1.0
    night_obs_count_p25: 7.0
    night_obs_count_p5: 3.0
    night_obs_count_p50: 13.0
    night_obs_count_p75: 20.0
    night_obs_count_p95: 39.0


## 6. LST anomaly and min-max normalization

Computed independently for Day and Night:

**Anomaly** = P90 LST − mean(P90 LST over POA) → UHI signal (°C)  
**Normalization** = (P90 LST − min_POA) / (max_POA − min_POA) → [0, 1]

In [8]:
def anomaly_and_norm(lst_p90, band_name, label):
    """
    Compute LST anomaly (relative to POA mean) and
    min-max normalization [0, 1] for a given P90 LST image.
    Returns (anomaly_image, norm_image, stats_dict).
    """
    roi_stats = lst_p90.reduceRegion(
        reducer=ee.Reducer.mean()
            .combine(ee.Reducer.min(), sharedInputs=True)
            .combine(ee.Reducer.max(), sharedInputs=True),
        geometry=roi, scale=1000, maxPixels=1e8, bestEffort=True,
    )

    poa_mean = ee.Number(roi_stats.get(f'{band_name}_mean'))
    poa_min  = ee.Number(roi_stats.get(f'{band_name}_min'))
    poa_max  = ee.Number(roi_stats.get(f'{band_name}_max'))

    anomaly = lst_p90.subtract(poa_mean).rename(f'{band_name}_anomaly')
    norm    = (
        lst_p90.subtract(poa_min)
        .divide(poa_max.subtract(poa_min))
        .rename(f'{band_name}_norm')
    )

    stats = {
        'mean': round(poa_mean.getInfo(), 2),
        'min':  round(poa_min.getInfo(),  2),
        'max':  round(poa_max.getInfo(),  2),
    }
    print(f'  {label}: min={stats["min"]}  mean={stats["mean"]}  max={stats["max"]} °C')
    return anomaly, norm, stats


print('Computing anomaly and normalization …')
lst_day_anomaly,   lst_day_norm,   stats_day   = anomaly_and_norm(lst_day_p90,   'lst_day_p90',   'Day  ')
lst_night_anomaly, lst_night_norm, stats_night = anomaly_and_norm(lst_night_p90, 'lst_night_p90', 'Night')

Computing anomaly and normalization …
  Day  : min=15.56  mean=28.57  max=40.4 °C
  Night: min=0.27  mean=17.3  max=32.87 °C


## 7. Summary statistics — all layers

In [9]:
all_layers = [
    (lst_day_p90,      'lst_day_p90',      'LST Day P90 (°C)'),
    (lst_night_p90,    'lst_night_p90',    'LST Night P90 (°C)'),
    (lst_day_anomaly,  'lst_day_p90_anomaly',   'LST Day Anomaly (°C)'),
    (lst_night_anomaly,'lst_night_p90_anomaly', 'LST Night Anomaly (°C)'),
    (lst_day_norm,     'lst_day_p90_norm',      'LST Day Norm (0-1)'),
    (lst_night_norm,   'lst_night_p90_norm',    'LST Night Norm (0-1)'),
]

print(f"{'Layer':<30} {'Min':>8} {'Mean':>8} {'Max':>8}")
print('-' * 58)
for img, band, label in all_layers:
    s = img.reduceRegion(
        reducer=ee.Reducer.min()
            .combine(ee.Reducer.mean(), sharedInputs=True)
            .combine(ee.Reducer.max(),  sharedInputs=True),
        geometry=roi, scale=1000, maxPixels=1e8, bestEffort=True,
    ).getInfo()
    vmin  = s.get(f'{band}_min',  s.get(band + '_min'))
    vmean = s.get(f'{band}_mean', s.get(band + '_mean'))
    vmax  = s.get(f'{band}_max',  s.get(band + '_max'))
    print(f"{label:<30} {vmin:>8.3f} {vmean:>8.3f} {vmax:>8.3f}")

Layer                               Min     Mean      Max
----------------------------------------------------------
LST Day P90 (°C)                 15.560   28.566   40.400
LST Night P90 (°C)                0.270   17.300   32.870
LST Day Anomaly (°C)            -13.006    0.000   11.834
LST Night Anomaly (°C)          -17.030    0.000   15.570
LST Day Norm (0-1)                0.000    0.524    1.000
LST Night Norm (0-1)              0.000    0.522    1.000


## 8. Visualization

In [12]:
# ── Visualization parameters ─────────────────────────────────────────────────
vis_day_p90 = {
    'min': 25, 'max': 55,
    'palette': ['#313695','#4575b4','#74add1','#abd9e9',
                '#fee090','#fdae61','#f46d43','#d73027','#a50026'],
}
vis_night_p90 = {
    'min': 18, 'max': 32,
    'palette': ['#313695','#4575b4','#74add1','#abd9e9',
                '#fee090','#fdae61','#f46d43','#d73027','#a50026'],
}
vis_anomaly = {
    'min': -8, 'max': 8,
    'palette': ['#2166ac','#92c5de','#f7f7f7','#f4a582','#b2182b'],
}
vis_norm = {
    'min': 0, 'max': 1,
    'palette': ['#440154','#31688e','#35b779','#6ece58','#fde725'],
}
vis_obs = {
    'min': 0, 'max': 30,
    'palette': ['#d7191c','#fdae61','#ffffbf','#a6d96a','#1a9641'],
}

# ── Map ───────────────────────────────────────────────────────────────────────
m = geemap.Map(center=[46.28, -94.30], zoom=11)
#m = geemap.Map(center=[-30.10, -51.16], zoom=11)
m.add_basemap('OpenStreetMap.Mapnik')

# Day layers
m.addLayer(lst_day_p90,     vis_day_p90,   'LST Day P90 (°C)',        True)
m.addLayer(lst_day_anomaly, vis_anomaly,   'LST Day Anomaly (°C)',    False)
m.addLayer(lst_day_norm,    vis_norm,      'LST Day Norm (0–1)',      False)
m.addLayer(day_obs,         vis_obs,       'Day Obs Count',           False)

# Night layers
m.addLayer(lst_night_p90,     vis_night_p90, 'LST Night P90 (°C)',      False)
m.addLayer(lst_night_anomaly, vis_anomaly,   'LST Night Anomaly (°C)',  False)
m.addLayer(lst_night_norm,    vis_norm,      'LST Night Norm (0–1)',    False)
m.addLayer(night_obs,         vis_obs,       'Night Obs Count',         False)

m.add_colorbar(
    vis_params=vis_day_p90,
    label='LST Day P90 (°C)',
    orientation='horizontal',
    position='bottomright',
    transparent_bg=True,
)
m.add_layer_control()
m

Map(center=[46.28, -94.3], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### Day − Night P90 difference

Shows the **diurnal thermal amplitude**: how much cooler each pixel is at night relative to day.  
Low values = small day–night difference = areas that never cool down (urban core, dense surfaces).  
This is a complementary diagnostic layer, not exported as a hazard input.

In [11]:
lst_day_night_diff = lst_day_p90.subtract(lst_night_p90).rename('day_night_diff')

diff_stats = lst_day_night_diff.reduceRegion(
    reducer=ee.Reducer.min()
        .combine(ee.Reducer.mean(), sharedInputs=True)
        .combine(ee.Reducer.max(),  sharedInputs=True),
    geometry=roi, scale=1000, maxPixels=1e8, bestEffort=True,
).getInfo()
print('Day − Night P90 difference (°C):')
print(f'  Min  : {diff_stats["day_night_diff_min"]:.2f}')
print(f'  Mean : {diff_stats["day_night_diff_mean"]:.2f}')
print(f'  Max  : {diff_stats["day_night_diff_max"]:.2f}')

vis_diff = {
    'min': 5, 'max': 20,
    'palette': ['#d73027','#f46d43','#fdae61','#ffffbf','#abd9e9','#74add1','#4575b4'],
}

m_diff = geemap.Map(center=[-30.10, -51.16], zoom=11)
m_diff.add_basemap('OpenStreetMap.Mapnik')
m_diff.addLayer(lst_day_night_diff, vis_diff, 'Day − Night P90 diff (°C)')
m_diff.add_colorbar(
    vis_params=vis_diff,
    label='Day − Night P90 (°C)',
    orientation='horizontal',
    position='bottomright',
    transparent_bg=True,
)
m_diff.add_layer_control()
m_diff

Day − Night P90 difference (°C):
  Min  : -4.85
  Mean : 11.27
  Max  : 28.81


Map(center=[-30.1, -51.16], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…

## 9. Export to Google Drive

Four GeoTIFFs exported at **1 km** (MODIS native resolution):

| Task | File |
|------|------|
| `task_day_p90` | `mod11a2_lst_day_p90_djf_2015_2024_poa` |
| `task_night_p90` | `mod11a2_lst_night_p90_djf_2015_2024_poa` |
| `task_day_norm` | `mod11a2_lst_day_norm_djf_2015_2024_poa` |
| `task_night_norm` | `mod11a2_lst_night_norm_djf_2015_2024_poa` |

> Exports go to `EE_exports/heat/` in your Google Drive.

In [ ]:
# --- Export MODIS LST layers to heat site input/ (local by default) ---
# Override: export GEE_EXPORT_MODE=drive

from gee_local_export import export_image_to_input

EXPORT_SCALE = 1000
EXPORT_CRS = "EPSG:4326"
EXPORT_FOLDER = "EE_exports/heat"

site_label = SITE_CONFIG["display_name"]
period_label = f"{SEASON_LABEL} {START_YEAR}-{END_YEAR}"
layer_names = SITE_CONFIG["layers"]

export_cfg = [
    (
        lst_day_p90,
        layer_names["modis_day_p90"],
        f"MODIS MOD11A2 P90 daytime LST {period_label} {site_label} (degC)",
    ),
    (
        lst_night_p90,
        layer_names["modis_night_p90"],
        f"MODIS MOD11A2 P90 nighttime LST {period_label} {site_label} (degC)",
    ),
    (
        lst_day_norm,
        layer_names["modis_day_norm"],
        f"MODIS MOD11A2 normalized P90 daytime LST {period_label} {site_label} (0-1)",
    ),
    (
        lst_night_norm,
        layer_names["modis_night_norm"],
        f"MODIS MOD11A2 normalized P90 nighttime LST {period_label} {site_label} (0-1)",
    ),
]

for image, filename, desc in export_cfg:
    export_image_to_input(
        image,
        filename=filename,
        region=roi,
        scale=EXPORT_SCALE,
        input_dir=INPUT_DIR,
        crs=EXPORT_CRS,
        description=Path(filename).stem,
        drive_folder=EXPORT_FOLDER,
    )

print("Exports complete. Files land under:", INPUT_DIR)


## 10. Check export status

## 11. Convert MOD11A2 LST P90 to COG and Generate Tiles

Publish the local MOD11A2 P90 LST rasters for web maps: COG + colorized XYZ tiles + value-encoded XYZ tiles for hover lookup.

Inputs:
- `sites/<site_slug>/data/input/mod11a2_lst_day_p90_djf_2015_2024_poa.tif`
- `sites/<site_slug>/data/input/mod11a2_lst_night_p90_djf_2015_2024_poa.tif`

Requires GDAL CLI (`gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`) and the matching color tables in `sites/<site_slug>/data/output/`.

In [12]:
# Convert MOD11A2 LST P90 GeoTIFFs to COG + visual tiles + value tiles.
from pathlib import Path
import os
import shutil
import subprocess

# Jupyter kernels often do not inherit Homebrew paths.
for extra_path in [
    "/opt/homebrew/bin",
    "/usr/local/bin",
    "/opt/homebrew/opt/gdal/bin",
    "/usr/local/opt/gdal/bin",
]:
    if Path(extra_path).exists() and extra_path not in os.environ.get("PATH", ""):
        os.environ["PATH"] = extra_path + os.pathsep + os.environ.get("PATH", "")


def resolve_gdal2tiles_python() -> str:
    gdal2tiles = shutil.which("gdal2tiles.py")
    if not gdal2tiles:
        raise RuntimeError(
            "gdal2tiles.py not found in PATH. Install GDAL (e.g. brew install gdal) "
            "or ensure /opt/homebrew/bin is on PATH for this kernel."
        )

    gdal2tiles_python = None
    with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip()
    if first_line.startswith("#!"):
        shebang_parts = first_line[2:].split()
        if shebang_parts:
            if shebang_parts[0].endswith("env") and len(shebang_parts) > 1:
                gdal2tiles_python = shutil.which(shebang_parts[1])
            else:
                gdal2tiles_python = shebang_parts[0]
    if not gdal2tiles_python:
        gdal2tiles_python = shutil.which("python3") or shutil.which("python")
    subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)
    return gdal2tiles_python


def publish_lst_layer(layer: dict, zoom: str = "8-15") -> None:
    in_tif = layer["in_tif"]
    out_dir = layer["out_dir"]
    colors_txt = layer["colors_txt"]
    slug = layer["slug"]
    label = layer["label"]

    cog_tif = out_dir / f"{slug}_cog.tif"
    colorized_tif = out_dir / f"{slug}_colorized.tif"
    value_encoded_tif = out_dir / f"{slug}_value_encoded_rgb.tif"
    tiles_dir = out_dir / "tiles_visual"
    value_tiles_dir = out_dir / "tiles_values"
    decode_txt = out_dir / f"{slug}_value_tiles_decode.txt"

    out_dir.mkdir(parents=True, exist_ok=True)
    if not in_tif.exists():
        raise FileNotFoundError(f"Missing input raster: {in_tif}")
    if not colors_txt.exists():
        raise FileNotFoundError(f"Missing color table: {colors_txt}")

    print(f"\nPublishing {label}")
    print("Input:", in_tif)
    print("Output dir:", out_dir)

    # 1) COG preserving raw LST values in degrees Celsius.
    subprocess.run([
        "gdal_translate", str(in_tif), str(cog_tif),
        "-of", "COG",
        "-ot", "Float32",
        "-co", "COMPRESS=DEFLATE",
        "-co", "RESAMPLING=NEAREST",
        "-co", "OVERVIEWS=AUTO",
    ], check=True)
    print("Created COG:", cog_tif)

    # 2) Colorized raster + visual XYZ tiles.
    subprocess.run([
        "gdaldem", "color-relief",
        str(cog_tif), str(colors_txt), str(colorized_tif),
        "-alpha",
    ], check=True)
    print("Created colorized raster:", colorized_tif)

    tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "gdal2tiles.py",
        "-r", "near",
        "-z", zoom,
        "--xyz",
        "-w", "none",
        str(colorized_tif),
        str(tiles_dir),
    ], check=True)
    print("Visual tiles:", tiles_dir)

    # 3) Value tiles for client-side hover.
    # Encodes Celsius at 0.01 C precision with +100 C offset plus 1 so encoded RGB value 0 can mean nodata.
    # Decode: encoded = R + 256*G + 65536*B; lst_celsius = ((encoded - 1) / 100) - 100; encoded == 0 => nodata.
    base_expr = (
        "numpy.where(numpy.isnan(A), 0, "
        "numpy.rint(numpy.clip(A + 100,0,167772.14)*100).astype(numpy.int64) + 1)"
    )
    subprocess.run([
        "gdal_calc.py",
        "-A", str(cog_tif),
        "--calc", f"bitwise_and({base_expr},255)",
        "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
        "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
        "--type", "Byte",
        "--NoDataValue", "0",
        "--overwrite",
        "--outfile", str(value_encoded_tif),
    ], check=True)

    value_tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "gdal2tiles.py",
        "-r", "near",
        "-z", zoom,
        "--xyz",
        "-w", "none",
        str(value_encoded_tif),
        str(value_tiles_dir),
    ], check=True)

    metadata = f"""{label} value tiles

Source raster: {in_tif.relative_to(PROJECT_ROOT)}
COG: {cog_tif.relative_to(PROJECT_ROOT)}
Visual tiles: {tiles_dir.relative_to(PROJECT_ROOT)}/{{z}}/{{x}}/{{y}}.png
Value tiles: {value_tiles_dir.relative_to(PROJECT_ROOT)}/{{z}}/{{x}}/{{y}}.png

Value tile encoding:
encoded = R + 256 * G + 65536 * B
if encoded == 0: value is nodata
else: lst_celsius = ((encoded - 1) / 100) - 100

LST values are MOD11A2 P90 land surface temperature in degrees Celsius, encoded at 0.01 C precision.
"""
    decode_txt.write_text(metadata, encoding="utf-8")

    print("Value tiles:", value_tiles_dir)
    print("Decode metadata:", decode_txt)
    print("Decode: encoded = R + 256*G + 65536*B; lst_celsius = ((encoded - 1) / 100) - 100; encoded 0 = nodata")


PROJECT_ROOT = HEAT_HAZARD_ROOT
resolve_gdal2tiles_python()

def resolve_modis_colors(layer_key: str) -> Path:
    """Prefer city/season-specific palette; fall back to shared day/night tables."""
    stem = SITE_CONFIG["layers"][layer_key].removesuffix(".tif")
    specific = STYLES_DIR / f"{stem}_colors.txt"
    if specific.exists():
        return specific
    shared_name = (
        "mod11a2_lst_day_p90_colors.txt"
        if "day" in layer_key
        else "mod11a2_lst_night_p90_colors.txt"
    )
    shared = STYLES_DIR / shared_name
    if shared.exists():
        print(f"Using shared color table for {layer_key}: {shared.name}")
        return shared
    raise FileNotFoundError(
        f"Missing color table for {layer_key}. Tried {specific} and {shared}."
    )


season_label = SITE_CONFIG.get("season_label") or SITE_CONFIG.get("season", "").upper()
year_span = f"{SITE_CONFIG.get('start_year', '')}-{SITE_CONFIG.get('end_year', '')}"

layers = [
    {
        "slug": SITE_CONFIG["layers"]["modis_day_p90"].removesuffix(".tif"),
        "label": f"MOD11A2 daytime LST P90 {season_label} {year_span}",
        "in_tif": INPUT_DIR / SITE_CONFIG["layers"]["modis_day_p90"],
        "out_dir": OUT_ROOT / SITE_CONFIG["layers"]["modis_day_p90"].removesuffix(".tif"),
        "colors_txt": resolve_modis_colors("modis_day_p90"),
    },
    {
        "slug": SITE_CONFIG["layers"]["modis_night_p90"].removesuffix(".tif"),
        "label": f"MOD11A2 nighttime LST P90 {season_label} {year_span}",
        "in_tif": INPUT_DIR / SITE_CONFIG["layers"]["modis_night_p90"],
        "out_dir": OUT_ROOT / SITE_CONFIG["layers"]["modis_night_p90"].removesuffix(".tif"),
        "colors_txt": resolve_modis_colors("modis_night_p90"),
    },
]

for layer in layers:
    publish_lst_layer(layer)


Using shared color table for modis_day_p90: mod11a2_lst_day_p90_colors.txt
Using shared color table for modis_night_p90: mod11a2_lst_night_p90_colors.txt

Publishing MOD11A2 daytime LST P90 JJA 2015-2024
Input: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/data/input/mod11a2_lst_day_p90_jja_2015_2024_plymouth.tif
Output dir: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_day_p90_jja_2015_2024_plymouth
Input file size is 14, 11
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_day_p90_jja_2015_2024_plymouth/mod11a2_lst_day_p90_jja_2015_2024_plymouth_cog.tif
0...10...20...30...40...50...60...70...80...90...100 - done.
Created colorized raster: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_day_p90_jja_2015_2024_plymouth/mod11a2_

Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40

Generating Overview Tiles:


...50...60...70...80...90...100 - done.
Visual tiles: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_day_p90_jja_2015_2024_plymouth/tiles_visual
0...10...20...30...40...50...60...70...80...90...100 - done.


...

.40...50...60...70...80...90...100 - done.
Visual tiles: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_night_p90_jja_2015_2024_plymouth/tiles_visual
0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...

Generating Overview Tiles:


70...80...90...100 - done.
Value tiles: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_night_p90_jja_2015_2024_plymouth/tiles_values
Decode metadata: /Users/admin/Desktop/OEF/geospatial-data/transformation/heat_hazard/sites/plymouth/out/mod11a2_lst_night_p90_jja_2015_2024_plymouth/mod11a2_lst_night_p90_jja_2015_2024_plymouth_value_tiles_decode.txt
Decode: encoded = R + 256*G + 65536*B; lst_celsius = ((encoded - 1) / 100) - 100; encoded 0 = nodata


In [ ]:
for name, task in tasks:
    status = task.status()
    state  = status.get('state', 'UNKNOWN')
    error  = status.get('error_message', '')
    print(f'{name}: {state}' + (f' — {error}' if error else ''))